# Evo1 DNA Sequence Generation and Scoring

This notebook demonstrates both tools that utilize Evo1 model. In the first section, we use `run_evo1_sample` to extend a short DNA prompt into a longer sequence using autoregressive sampling. In the second section, we use `run_evo1_score` to compute log-likelihood and perplexity metrics for a set of DNA sequences, which can be used to assess sequence quality or compare design candidates.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("evo1")
display_overview("evo1")
display_docs_section("evo1", "Background")

# Evo1

> Evo1 is a 7-billion parameter DNA language model built on the [StripedHyena](https://en.wikipedia.org/wiki/State_space_model#Deep_learning) architecture, trained on 2.7 million [prokaryotic](https://en.wikipedia.org/wiki/Prokaryote) and [phage](https://en.wikipedia.org/wiki/Bacteriophage) genomes from the OpenGenome dataset. This tool performs autoregressive DNA sequence generation from prompts and optionally scores generated sequences by mean log-probability.

## Background

**What does this tool measure/predict?**
Evo1 generates novel DNA sequences by learning the statistical patterns of prokaryotic and phage genomes. It can also score sequences by log-likelihood, providing a measure of how "natural" a generated sequence appears relative to the training distribution.

**Why is this important?**
Generative DNA models enable de novo design of biological sequences; genes, [operons](https://en.wikipedia.org/wiki/Operon), [CRISPR](https://en.wikipedia.org/wiki/CRISPR) systems, and [mobile genetic elements](https://en.wikipedia.org/wiki/Transposable_element); without requiring template sequences. By learning the grammar of genomes at single-nucleotide resolution, Evo1 can propose functional DNA that respects codon usage, regulatory signals, and structural constraints.

**Scientific foundation:**
Evo1 uses the StripedHyena architecture, a hybrid state-space/attention model that processes DNA at single-nucleotide resolution. Unlike transformer-only models, StripedHyena supports efficient long-range sequence modeling up to 131k tokens. The model is trained with a standard autoregressive (next-token prediction) objective on raw genomic DNA.

## Available tools

In [2]:
display_available_tools("evo1")

- **`run_evo1_sample()`** — Sample DNA sequences using Evo1 language model
- **`run_evo1_score()`** — Score DNA sequences using Evo1 language model

### `run_evo1_sample`

Evo1 generates DNA sequences autoregressively from a starting prompt, extending a sequence nucleotide-by-nucleotide according to the model's learned distribution over prokaryotic and phage genomes. The `temperature` and `top_k` parameters control the diversity of the output: lower values produce more conservative, high-confidence extensions while higher values allow more exploratory generation. Fine-tuned checkpoints are also available for CRISPR and transposon-specific generation.

In [3]:
from proto_tools import (
    Evo1SampleInput,
    Evo1SampleConfig,
    run_evo1_sample,
)

In [4]:
# Dipslay docs
display_api_reference("evo1", "input", "run_evo1_sample")

# Create a simple Evo1 Input with one starting prompt
inputs = Evo1SampleInput(prompts=["ACGTACGT"])

**Input** — `CausalModelSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `prompts` | `List[string]` | required | Prompt sequences to condition generation on. Can be provided as a single string or a list of strings. |

In [5]:
# Display config docs
display_api_reference("evo1", "config", "run_evo1_sample")

# Create a simple Evo1Sample config | see docs below for all fields
config = Evo1SampleConfig(
    model_name="evo-1-8k-base",
    num_tokens=200,
    temperature=0.1,
    top_k=4,
    prepend_prompt=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `Evo1SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_name` | `enum` | `evo-1-8k-base` | Evo1 model checkpoint to use. Available options: `evo-1-8k-base`, `evo-1-131k-base`, `evo-1-8k-crispr`, `evo-1-8k-transposon` |
| `top_k` | `integer` | `4` | Number of top tokens to consider for sampling. |
| `num_tokens` | `integer` | `100` | Number of tokens to generate per prompt. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `prepend_prompt` | `boolean` | `False` | Whether to include the input prompt at the start of each generated sequence. |
| `temperature` | `number` | `1.0` | Sampling temperature controlling randomness. Higher values produce more diverse outputs; lower values produce more conservative, high-confidence sequences. |
| `top_p` | `number` | `1.0` | Nucleus sampling threshold. Only tokens whose cumulative probability mass reaches this threshold are considered during sampling. Lower values restrict sampling to higher-probability tokens. |
| `batch_size` | `integer` | `1` | Number of prompts to process simultaneously on GPU. Larger batches improve throughput but use more GPU memory; reduce if encountering out-of-memory errors. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the sampling function
sample_result = run_evo1_sample(inputs, config)

Running run_evo1_sample [00:00]

In [7]:
display_api_reference("evo1", "output", "run_evo1_sample")

# Select the prompt and corresponding generated output sequence and show some results
prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} nucleotides")
if sample_result.scores:
    score = sample_result.scores[0]
    print(f"Avg log-likelihood: {score['avg_log_likelihood']:.4f}")
    print(f"Perplexity:         {score['perplexity']:.4f}")

**Output** — `Evo1SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `array` |  | Scoring metrics per sequence, including log\_likelihood, avg\_log\_likelihood, and perplexity. |
| `sequences` | `List[string]` | required | Generated DNA sequences. |

Prompt:       ACGTACGT
Generated:    ACGTACGTCGGCGACGAGGTCGTCGGCGTCGTCGAGGAGGTCGGCAAGGACGTCGCCACGGTCGCCCTCGGCTACCGCGA...
Total length: 208 nucleotides
Avg log-likelihood: -2.2720
Perplexity:         9.6992


### `run_evo1_score`

Evo1 scores sequences by computing the autoregressive log-likelihood at each position, measuring how well each nucleotide follows from its preceding context according to the model. The resulting perplexity is a single number summarizing sequence naturalness: lower perplexity indicates that the sequence more closely matches the statistical patterns of prokaryotic and phage genomes in the training data. This is useful for ranking generated candidates, filtering out low-quality designs, or comparing the naturalness of wild-type versus engineered sequences.

In [8]:
from proto_tools import (
    Evo1ScoringInput,
    Evo1ScoringConfig,
    run_evo1_score,
)

In [9]:
# Display docs
display_api_reference("evo1", "input", "run_evo1_score")

# Score two short DNA sequences
score_inputs = Evo1ScoringInput(sequences=["ATGCGTAAA", "ATGCGTAAATTT"])

**Input** — `CausalModelScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Sequences to score. Can be provided as a single string or a list of strings. |

In [10]:
# Display config docs
display_api_reference("evo1", "config", "run_evo1_score")

# Create a simple Evo1Scoring config | see docs above for all fields
score_config = Evo1ScoringConfig(
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `Evo1ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_name` | `enum` | `evo-1-8k-base` | Evo1 model checkpoint to use. Default: `"evo-1-8k-base"`. Available options: `evo-1-8k-base`, `evo-1-131k-base`, `evo-1-8k-crispr`, `evo-1-8k-transposon` |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `batch_size` | `integer` | `1` | Number of sequences to process simultaneously on GPU. Larger batches improve throughput but use more GPU memory. |
| `return_logits` | `boolean` | `False` | Whether to include per-position logits in the output. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [11]:
# Run the scoring function
score_result = run_evo1_score(score_inputs, score_config)

Running run_evo1_score [00:00]

In [12]:
# Display output docs
display_api_reference("evo1", "output", "run_evo1_score")

# Show scoring results for each input sequence
for i, seq in enumerate(score_inputs.sequences):
    score = score_result.scores[i]
    print(f"Sequence {i + 1}: {seq}")
    print(f"    Log-likelihood:     {score.log_likelihood:.3f}")
    print(f"    Avg log-likelihood: {score.avg_log_likelihood:.3f}")
    print(f"    Perplexity:         {score.perplexity:.3f}")

**Output** — `CausalModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `List[CausalModelScoringMetrics]` | required | List of scoring outputs, one per input sequence. Each entry contains metrics (log\_likelihood, avg\_log\_likelihood, perplexity) and optional per-position logits. |
| `metrics` | `Dict[string, number]` |  | Dictionary of scalar scoring metrics. |
| `logits` | `array` |  | Optional per-position logits array. |
| `vocab` | `array` |  | Optional token ordering for logits; logits\[:, j] corresponds to vocab\[j]. |
| `per_position_metrics` | `Dict[string, List[number]]` |  | Optional per-position scoring metrics, keyed by metric name. Each value is a list of length equal to the input sequence, with `None` at positions where that metric is not available. |

Sequence 1: ATGCGTAAA
    Log-likelihood:     -13.398
    Avg log-likelihood: -1.489
    Perplexity:         4.431
Sequence 2: ATGCGTAAATTT
    Log-likelihood:     -17.062
    Avg log-likelihood: -1.422
    Perplexity:         4.145


## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for use in downstream analysis pipelines. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sample_result.export(name="evo1_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'evo1_sequences.fasta'}")

score_result.export(name="evo1_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'evo1_scores.csv'}")

Generated sequences exported to example_output/evo1_sequences.fasta
Scores exported to example_output/evo1_scores.csv
